# SMS Spam Dataset - Exploratory Data Analysis

?? Notebook ???? `dataset/sms_spam.csv` ????????????????????????


## ????
- ?? `dataset/sms_spam.csv` ?????????? `python ingest/download_dataset.py` ?????
- ?????????????`pip install -r requirements.txt`?
- ????????????????????????


In [ ]:
from pathlib import Path
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams['figure.figsize'] = (10, 6)
try:
    plt.style.use('seaborn-v0_8')
except OSError:
    pass

pd.set_option('display.max_colwidth', 120)


In [ ]:
DATASET_PATH = Path('dataset/sms_spam.csv')

if not DATASET_PATH.exists():
    raise FileNotFoundError('??? dataset/sms_spam.csv????? `python ingest/download_dataset.py` ??????')

df = pd.read_csv(
    DATASET_PATH,
    header=None,
    names=['label', 'message'],
)

df.head()


### ?????


In [ ]:
row_count, col_count = df.shape
print(f'????: {row_count:,}')
print(f'??: {list(df.columns)}')

df.describe(include='all').transpose()


### ????


In [ ]:
class_counts = df['label'].value_counts().sort_index()
class_ratio = (class_counts / len(df)).rename('ratio')

pd.concat([class_counts.rename('count'), class_ratio], axis=1)


In [ ]:
ax = class_counts.plot(kind='bar', color=['#4e79a7', '#f28e2b'])
ax.set_title('Class distribution')
ax.set_xlabel('label')
ax.set_ylabel('count')
for container in ax.containers:
    ax.bar_label(container, label_type='edge', padding=3)
plt.tight_layout()
plt.show()


### ??????


In [ ]:
df['message_length'] = df['message'].str.len()

df['message_length'].describe()


In [ ]:
length_summary = (
    df.groupby('label')['message_length']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .round({'mean': 2, 'median': 2})
)
length_summary


In [ ]:
plt.figure()
(df['message_length']).plot(kind='hist', bins=50, color='#59a14f', alpha=0.85)
plt.title('Distribution of message length (characters)')
plt.xlabel('characters per message')
plt.ylabel('frequency')
plt.tight_layout()
plt.show()


### ????


In [ ]:
token_pattern = re.compile(r"[A-Za-z0-9']+")

def tokenize(text: str):
    return token_pattern.findall(text.lower())

overall_counter = Counter()
for text in df['message']:
    overall_counter.update(tokenize(text))

top_overall = pd.DataFrame(overall_counter.most_common(15), columns=['token', 'count'])
top_overall


In [ ]:
label_tokens = []
for label, subset in df.groupby('label'):
    counter = Counter()
    for text in subset['message']:
        counter.update(tokenize(text))
    label_tokens.append(
        pd.DataFrame(counter.most_common(10), columns=['token', 'count']).assign(label=label)
    )

top_by_label = pd.concat(label_tokens, ignore_index=True)
top_by_label


### ????


In [ ]:
for label, subset in df.groupby('label'):
    print(f'???? - {label}')
    display(subset[['message', 'message_length']].sample(3, random_state=42))
